In [16]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score

# ==========================================
# 1. KONFIGURASI PARAMETER (Terpisah)
# ==========================================
PARAMS = {
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
    'alphas': np.logspace(-4, 2, 70),
    'l1_ratio': 0.5  # Khusus untuk Elastic Net
}


In [ ]:
# ==========================================
# 2. CLASS REGULARIZATION COMPARISON
# ==========================================
class RegularizationTrainer:
    def __init__(self, params):
        self.params = params
        self.scaler = StandardScaler()
        self.models = {}
        self.results = None
        self.coef_df = None

    def prepare_data(self):
        """Memuat dataset, splitting, dan scaling."""
        diabetes = load_diabetes()
        X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
        y = diabetes.target
        
        # Splitting
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=self.params['test_size'], 
            random_state=self.params['random_state']
        )
        
        # Scaling
        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_test_scaled = self.scaler.transform(X_test)
        self.y_train = y_train
        self.y_test = y_test
        self.feature_names = diabetes.feature_names

    def train_models(self):
        """Inisialisasi dan melatih semua model dengan CV."""        
        # OLS
        self.models['OLS'] = LinearRegression()
        
        # Ridge CV
        self.models['Ridge'] = RidgeCV(
            alphas=self.params['alphas']
        )
        
        # Lasso CV
        self.models['Lasso'] = LassoCV(
            alphas=self.params['alphas'], 
            cv=self.params['cv_folds'], 
            random_state=self.params['random_state']
        )
        
        # Elastic Net CV
        self.models['ElasticNet'] = ElasticNetCV(
            alphas=self.params['alphas'], 
            l1_ratio=self.params['l1_ratio'], 
            cv=self.params['cv_folds'], 
            random_state=self.params['random_state']
        )
        
        # Fitting all models
        for name, model in self.models.items():
            model.fit(self.X_train_scaled, self.y_train)

    def evaluate(self):
        """Menghitung R2 Score dan mengumpulkan koefisien."""
        metrics = []
        coefficients = {'Fitur': self.feature_names}
        
        for name, model in self.models.items():
            # Hitung R2
            y_pred = model.predict(self.X_test_scaled)
            score = r2_score(self.y_test, y_pred)
            
            # Ambil Alpha Terbaik
            best_alpha = getattr(model, 'alpha_', 'N/A')
            
            metrics.append({
                'Model': name, 
                'R2 Score': score, 
                'Best Alpha': best_alpha
            })
            
            # Ambil Koefisien
            coefficients[name] = model.coef_
            
        self.results = pd.DataFrame(metrics)
        self.coef_df = pd.DataFrame(coefficients)

    def display_results(self):
        """Menampilkan hasil ke layar."""
        print("\n" + "="*30)
        print("HASIL PERFORMA MODEL")
        print("="*30)
        print(self.results)
        
        print("\n" + "="*30)
        print("PERBANDINGAN KOEFISIEN")
        print("="*30)
        print(self.coef_df)

In [18]:
# ==========================================
# 3. EKSEKUSI PROGRAM
# ==========================================
if __name__ == "__main__":
    # Inisialisasi trainer dengan parameter terpisah
    trainer = RegularizationTrainer(PARAMS)
    
    # Jalankan alur kerja
    trainer.prepare_data()
    trainer.train_models()
    trainer.evaluate()
    trainer.display_results()

Sedang melatih model (ini mungkin memakan waktu beberapa saat)...

HASIL PERFORMA MODEL
        Model  R2 Score Best Alpha
0         OLS  0.452603        N/A
1       Ridge  0.454366   1.221677
2       Lasso  0.471038   1.492496
3  ElasticNet  0.461141   0.201534

PERBANDINGAN KOEFISIEN
  Fitur        OLS      Ridge      Lasso  ElasticNet
0   age   1.753758   1.816017   0.053450    1.890461
1   sex -11.511809 -11.435368  -8.248885   -9.882390
2   bmi  25.607121  25.749240  26.206473   24.328641
3    bp  16.828872  16.717025  15.176519   15.425506
4    s1 -44.448856 -33.094546  -5.161475   -5.587204
5    s2  24.640954  15.832268  -0.000000   -3.787195
6    s3   7.676978   2.675542 -11.171354   -8.942693
7    s4  13.138784  11.540881   0.000000    7.005844
8    s5  35.161195  30.766150  22.173946   19.192871
9    s6   2.351364   2.477470   1.859795    3.608240
